# ESPIRiT: is a 20x20 ACS enough?

The brain maps on disk were made by `scripts/make_espirit_smaps.py` with
`--acs 20`, and the recon nets that consume them underperform an E2E-VarNet,
which estimates its own maps. This notebook estimates maps at the shipped ACS
and at a large one on the same slice and compares the **coil-combined images**
`x = sum_c conj(s_c) c_c` -- the ground truth every net is trained to produce.

The maths lives in `scripts/check_espirit_acs.py` and is imported, not copied;
this notebook is the interactive front end to it. For a headless run over a
whole volume use the script directly:

    python -m scripts.check_espirit_acs --anatomy brain --split val --acs 20,48,96

**What the numbers mean**

| column | what it says | bad looks like |
|---|---|---|
| `rows/cols` | shape of the ESPIRiT calibration matrix | `< 1`: no null space is actually estimated |
| `support` / `brain unc` | where the maps are nonzero, and how much brain they miss | any brain uncovered is unrecoverable by any net |
| `\|RSS-1\|` | `operators/noise.py::mri_awgn` assumes `sum_c \|s_c\|^2 = 1` | not ~0 |
| `residual` | `\|\| c - s x \|\| / \|\| c \|\|`, the part of the coil data the SENSE model cannot represent | a floor no net can beat |
| `retention` | `\|\|x\|\| / \|\|RSS\|\|` | `< 1`: oversmoothed maps combining the coils incoherently |
| `phase coh` | smoothness of `arg x` | low at EVERY acs means the coil-0 phase reference is the problem, not the ACS |

Maps are defined only up to a per-pixel phase, so nothing here diffs maps
against maps -- that would measure the convention. Everything is scored through
the images the maps produce.

In [ ]:
%matplotlib inline
import os
import pathlib
import sys
import time

import numpy as np
import torch
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if not (ROOT / "physics").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from operators.fourier import ifftc
from physics.smaps import espirit, espirit_soft
from scripts.check_espirit_acs import (
    KSPACE_ROOTS, calib_shape, cg_sense_nrmse, compare, crop_readout, draw,
    load_slices, map_metrics, pick_volume, subspace_residual,
)

print("root", ROOT)

In [ ]:
# ===================== WHAT TO COMPARE -- TUNE =====================
ANATOMY  = "brain"
SPLIT    = "val"
VOLUME   = None            # None -> first matching volume under KSPACE_ROOTS
SLICE    = "mid"           # "mid", "all", or e.g. "4,8,12"

# FIRST entry is treated as the shipped setting, LAST as the reference.
ACS_LIST = [(20, 20), (48, 48), (96, 96)]

KERNEL_SIZE     = 8        # the defaults are make_espirit_smaps.py's
THRESH_EIG      = 0.95
THRESH_ROWSPACE = 0.05
MAXIT           = 100      # espirit power-method iterations

CROP_READOUT = False       # drop brain's 2x readout oversampling first
MASK_THRESH  = 0.05        # brain mask is RSS > this, with RSS scaled to max 1
PHASE_KS     = 5           # phase-coherence boxcar side
DIFF_SCALE   = 0.2         # difference panels use +- this x the RSS window

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# ===================================================================
print(DEVICE)

## Load one slice

The slice is scaled so that `max(RSS) = 1`. Every metric below is a ratio and
does not care, but it makes the display windows and the CG-SENSE `lamda`
comparable between volumes.

In [ ]:
path = VOLUME or pick_volume(KSPACE_ROOTS[ANATOMY].format(split=SPLIT), ANATOMY, 0)
kvol, sl_idx = load_slices(path, SLICE)
if CROP_READOUT:
    kvol = crop_readout(kvol)

SI = 0                                    # which of the loaded slices to work on
C, Nx, Ny = kvol.shape[1], kvol.shape[-2], kvol.shape[-1]

k = kvol[SI:SI + 1].to(DEVICE)
coil = ifftc(k)
rss = coil.abs().pow(2).sum(1, keepdim=True).sqrt()
scale = rss.amax().clamp_min(1e-12)
k, coil, rss = k / scale, coil / scale, rss / scale
brain = rss[:, 0] > MASK_THRESH

print(f"{os.path.basename(path)}   slice {sl_idx[SI]} of {sl_idx}")
print(f"  grid {(Nx, Ny)}   coils {C}   "
      f"brain {float(brain.float().mean()):.1%} of FOV"
      f"{'   (readout oversampling cropped)' if CROP_READOUT else ''}")

## 1. The calibration geometry, before computing anything

ESPIRiT splits the calibration matrix's row space from its null space. The
matrix has `(ax-ks+1)(ay-ks+1)` rows -- one per ACS patch -- and `ks^2 * C`
columns. When there are fewer rows than columns there is no estimated null
space at all: the SVD returns at most `rows` singular vectors spanning a
`cols`-dimensional kernel space, and the retained subspace is whatever those
few patches happened to span.

This cell computes nothing and is the fastest way to find out that a setting
was never going to work.

In [ ]:
cols = KERNEL_SIZE ** 2 * C
print(f"calibration matrix   cols = ks^2 x C = {cols}")
print(f"  {'acs':>9}{'rows':>8}{'rows/cols':>11}{'map res (px)':>15}   verdict")
for acs in ACS_LIST:
    rows, _ = calib_shape(acs, KERNEL_SIZE, C)
    res = f"{Nx / max(acs[0], 1):.0f} x {Ny / max(acs[1], 1):.0f}"
    verdict = ("IMPOSSIBLE -- the kernel does not fit in the ACS" if rows == 0
               else "UNDERDETERMINED -- no null space is actually estimated"
               if rows < cols else "ok" if rows >= 2 * cols else "marginal")
    print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{rows:>8}{rows / cols:>11.2f}"
          f"{res:>15}   {verdict}")

need = KERNEL_SIZE - 1 + int(np.ceil(np.sqrt(cols)))
print(f"  square ACS needs >= {need} per side for rows >= cols at "
      f"ks={KERNEL_SIZE}, C={C}")

## 2. Estimate and score

`runs` keeps each map set and its coil-combined image, so later cells (and you)
can poke at them.

**Memory.** The kernel images are `coils x retained kernels x the full grid`,
complex, and the retained-kernel count grows with the ACS -- gigabytes per
slice on brain's 640x320 grid, which is why the generator chunks. An OOM here
is caught and reported per setting rather than losing the whole run; if one
fails, set `CROP_READOUT = True` or drop the largest ACS and re-run.

In [ ]:
runs = []
print(f"  {'acs':>9}{'support':>9}{'brain unc':>11}{'|RSS-1|':>10}{'residual':>10}"
      f"{'retention':>11}{'p5 |x|/RSS':>12}{'phase coh':>11}{'sec':>7}")

for acs in ACS_LIST:
    tag = f"{acs[0]}x{acs[1]}"
    rows, _ = calib_shape(acs, KERNEL_SIZE, C)
    if rows == 0:
        print(f"  {tag:>9}  skipped: no patch fits -- kernel_size "
              f"{KERNEL_SIZE} exceeds the ACS")
        runs.append(None)
        continue

    ub = C * min(rows, cols) * Nx * Ny * 8 / 2 ** 30
    t0 = time.time()
    try:
        sm = espirit(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                     thresh_rowspace=THRESH_ROWSPACE, thresh_eig=THRESH_EIG,
                     maxit=MAXIT)
    except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  {tag:>9}  FAILED: {type(e).__name__}: "
              f"{str(e).splitlines()[0][:80]}")
        print(f"            kernel images need up to {ub:.1f} GB here")
        runs.append(None)
        continue

    r = map_metrics(sm, coil, rss, brain, PHASE_KS)
    r["sm"] = sm
    runs.append(r)
    print(f"  {tag:>9}{r['support']:>9.1%}{r['uncovered']:>11.2%}"
          f"{r['rss_err']:>10.1e}{r['residual']:>10.4f}{r['retention']:>11.4f}"
          f"{r['p5']:>12.3f}{r['phase_coh']:>11.3f}{time.time() - t0:>7.1f}")

live = [(a, r) for a, r in zip(ACS_LIST, runs) if r is not None]

## 3. The actual question: do the coil-combined images differ?

Two numbers, because they answer different things. The **magnitude** NRMSE is
what a magnitude-domain metric (PSNR/SSIM on `|x|`) would see. The **complex**
one removes a single global phase first -- the whole image rotating is a
convention difference and harmless; anything left is a per-pixel phase
disagreement, which a complex-valued net sees as structure.

In [ ]:
if len(live) >= 2:
    ref_acs, ref = live[-1]
    print(f"vs the acs {ref_acs[0]}x{ref_acs[1]} reference, over the brain:")
    print(f"  {'acs':>9}{'|x| NRMSE':>12}{'complex NRMSE':>16}")
    for acs, r in live[:-1]:
        mag, cpx = compare(r["x"], ref["x"], brain)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{mag:>12.4f}{cpx:>16.4f}")
else:
    print("need at least two surviving settings to compare")

## 4. Look at them

Every magnitude panel shares ONE window taken from the RSS of this slice, and
every difference panel shares one symmetric window at `DIFF_SCALE` times that.
Nothing is autoscaled per panel: a map set that loses 30% of the signal has to
LOOK 30% darker instead of being renormalised back into agreement. A blank
difference panel is therefore ambiguous on its own, so each one reports its
peak in the title.

Rows: coil-combined magnitude, support, `|s|` for coil 0, `arg x`, and the
difference against the RSS. The bottom-left panel is the shipped ACS minus the
reference ACS -- literally the question this notebook asks.

In [ ]:
fig = draw(rss, brain, runs, ACS_LIST, coil,
           f"{os.path.basename(path)}  slice {sl_idx[SI]}   ks={KERNEL_SIZE} "
           f"eig={THRESH_EIG} rowspace={THRESH_ROWSPACE}",
           DIFF_SCALE)

## 5. The maps used the way training uses them

A CG-SENSE reconstruction at `R`, scored against the fully sampled RSS inside
the brain. This is the end-to-end consequence of everything above: an unrolled
net handed these maps inherits whatever the forward model gets wrong.

`lamda` is not optional -- at `lamda = 0` the normal operator is singular
wherever the maps vanish and CG wanders in that null space.

In [ ]:
RECON_R         = 4
RECON_ACS_LINES = 24
RECON_LAMDA     = 1.0e-3
RECON_ITERS     = 64

print(f"CG-SENSE R={RECON_R} (acs_lines={RECON_ACS_LINES}, "
      f"lamda={RECON_LAMDA:g}), NRMSE vs fully sampled RSS:")
for acs, r in live:
    try:
        e = cg_sense_nrmse(r["sm"], k, rss, brain, RECON_R, RECON_ACS_LINES,
                           RECON_LAMDA, RECON_ITERS)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{e:>12.4f}")
    except Exception as exc:
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}  skipped: {exc}")

## 6. Is it the calibration, or the model?

A single map per coil says the coil data lies in a rank-1 subspace at every
pixel. Where that is false -- aliasing from outside the FOV, motion, fat/water
chemical shift -- no amount of calibration fixes it, and a second set of maps
does. `espirit_soft` returns `M` maps and this scores the same residual against
their span.

If the 1-map residual is high and the 2-map one is not, the model is the
problem and a bigger ACS will not help. That is also one of the things an
E2E-VarNet's learned maps can absorb.

In [ ]:
SOFT_MAPS = 2

print(f"  {'acs':>9}{'1-map':>10}{str(SOFT_MAPS) + '-map':>10}")
for acs, r in live:
    try:
        sms = espirit_soft(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                           thresh_rowspace=THRESH_ROWSPACE,
                           thresh_eig=THRESH_EIG, num_maps=SOFT_MAPS)
        rs = subspace_residual(sms, coil, brain)
        del sms
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{r['residual']:>10.4f}{rs:>10.4f}")
    except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}  skipped: {type(e).__name__}")

## 7. Sweep

`THRESH_EIG` sets the support and the ACS sets how much calibration data the
kernel sees. These two move the residual most. This is the slow cell -- one
full ESPIRiT per row.

In [ ]:
SWEEP_ACS = [(20, 20), (32, 32), (48, 48), (64, 64)]
SWEEP_EIG = [0.90, 0.95, 0.98]

print(f"  {'acs':>9}{'eig':>7}{'support':>10}{'brain unc':>11}"
      f"{'residual':>10}{'retention':>11}")
for acs in SWEEP_ACS:
    for eig in SWEEP_EIG:
        try:
            S = espirit(k, acs_size=acs, kernel_size=KERNEL_SIZE,
                        thresh_rowspace=THRESH_ROWSPACE, thresh_eig=eig,
                        maxit=MAXIT)
        except (torch.cuda.OutOfMemoryError, RuntimeError, MemoryError) as e:
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{eig:>7.2f}"
                  f"   skipped: {type(e).__name__}")
            continue
        m = map_metrics(S, coil, rss, brain, PHASE_KS)
        print(f"  {str(acs[0]) + 'x' + str(acs[1]):>9}{eig:>7.2f}"
              f"{m['support']:>10.1%}{m['uncovered']:>11.2%}"
              f"{m['residual']:>10.4f}{m['retention']:>11.4f}")
        del S, m

## How to read it

**The residual and retention barely move between 20x20 and the large ACS.**
The ACS is not what is hurting the nets. Look at the phase-coherence column and
at section 6 -- if the 2-map residual is much lower, the single-map model is the
ceiling; if phase coherence is low everywhere, the coil-0 phase reference in
`physics/smaps.py::espirit` is putting noise into the ground-truth phase wherever
coil 0 is dark.

**The residual drops / retention rises with the larger ACS.** The shipped maps
are calibration-starved. Regenerate with a larger ACS into a NEW directory:

    python scripts/make_espirit_smaps.py --anatomy brain --split train --acs 48

and point `smap_root` at it. That script rewrites `image` as well as `smaps`,
which is the point -- the coil-combined ground truth is a function of the maps,
and keeping the old one would leave a file whose ground truth is not what its
own operator produces.

Either way, re-run this on a few volumes (`VOLUME = "..."`) before committing to
a regeneration: coil counts and anatomy placement vary across fastMRI brain, and
the calibration table's verdict depends on `C`.